### Imports & configuration

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, date

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType,
    DateType
)

SOURCE_BASE_PATH = "abfss://raw@fintechdllasya.dfs.core.windows.net/customers/"
BRONZE_TABLE = "dbx_fintech_data_platform.bronze.customers"
INGESTION_LOG_TABLE = "dbx_fintech_data_platform.metadata.ingestion_log"

PIPELINE_NAME = "customer_bronze_ingestion"

log_schema = StructType([
    StructField("pipeline_name", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("source_date", DateType(), True),
    StructField("target_table", StringType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", LongType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("completed_at", TimestampType(), True),
    StructField("error_message", StringType(), True)
])

### Read the source file

In [0]:
source_path = f"{SOURCE_BASE_PATH}2026-08-18/customers.csv"

customer_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_path)
)

print("Source:", source_path)
print("Rows:", customer_df.count())

### Add Bronze metadata

In [0]:
bronze_df = (
    customer_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "_source_date",
        F.to_date(
            F.regexp_extract(
                F.col("_metadata.file_path"),
                r"/customers/(\d{4}-\d{2}-\d{2})/",
                1
            )
        )
    )
)

display(bronze_df.limit(10))

### Idempotency check

In [0]:
existing_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM {INGESTION_LOG_TABLE}
    WHERE source_file = '{source_path}'
      AND status = 'SUCCESS'
""").collect()[0]["cnt"]

if existing_count > 0:
    print("SKIPPED: Source file already processed.")
else:
    print("NEW FILE: Ready for ingestion.")

### Write only when the file is new

In [0]:
if existing_count == 0:

    rows_processed = bronze_df.count()

    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

    print(f"SUCCESS: Loaded {rows_processed} rows into {BRONZE_TABLE}")

else:
    print("SKIPPED: Bronze write not required.")

### Log the successful ingestion

In [0]:
if existing_count == 0:

    rows_processed = bronze_df.count()

    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

    log_data = [(
        PIPELINE_NAME,
        source_path,
        bronze_df.select("_source_date").first()["_source_date"],
        BRONZE_TABLE,
        "SUCCESS",
        rows_processed,
        datetime.now(),
        datetime.now(),
        None
    )]

    log_df = spark.createDataFrame(
        log_data,
        schema=log_schema
    )

    log_df.write.mode("append").saveAsTable(
        INGESTION_LOG_TABLE
    )

    print(f"SUCCESS: Loaded and logged {rows_processed} rows.")

else:
    print("SKIPPED: Source file already processed.")

### Final Bronze validation

In [0]:
%sql

SELECT
    _source_date,
    COUNT(*) AS record_count,
    COUNT(DISTINCT customer_id) AS unique_customers
FROM dbx_fintech_data_platform.bronze.customers
GROUP BY _source_date
ORDER BY _source_date;

In [0]:
%sql

SELECT
    source_date,
    status,
    rows_processed,
    source_file
FROM dbx_fintech_data_platform.metadata.ingestion_log
ORDER BY source_date;

In [0]:
from datetime import datetime, date

log_data = [(
    PIPELINE_NAME,
    "abfss://raw@fintechdllasya.dfs.core.windows.net/customers/2026-08-17/customers.csv",
    date(2026, 8, 17),
    BRONZE_TABLE,
    "SUCCESS",
    100000,
    datetime.now(),
    datetime.now(),
    None
)]

log_df = spark.createDataFrame(
    log_data,
    schema=log_schema
)

log_df.write.mode("append").saveAsTable(
    INGESTION_LOG_TABLE
)

In [0]:
%sql

SELECT
    source_date,
    status,
    rows_processed
FROM dbx_fintech_data_platform.metadata.ingestion_log
ORDER BY source_date;